In [1]:
import pandas as pd
import pandas as pd
import numpy as np
from scipy.stats import hmean

df_xgb = pd.read_csv('XGBoost_metrics.csv')
df_lstm = pd.read_csv('LSTM_metrics.csv')
df_ridge = pd.read_csv('ridge_metrics.csv')
persistence_df = pd.read_csv('persistence_rmse.csv')

In [2]:
df_xgb

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,XGBoost,1.064968,0.067575,0.015356,0.868305,0.044141,0.001948


In [3]:
df_lstm

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,LSTM,1003.371662,5.492276,0.017471,0.856802,0.046028,0.002119


In [4]:
df_ridge

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,Ridge,0.058098,0.00539,0.027568,0.809839,0.053042,0.002813


In [5]:
persistence_df

,Model,RMSE
0,Persistence Model,0.07137


In [6]:
final_df = pd.concat([df_xgb, df_lstm, df_ridge], ignore_index=True)
final_df

,Model,Training Time (s),Prediction Time (s),MAE,R-squared,RMSE,MSE
0,XGBoost,1.064968,0.067575,0.015356,0.868305,0.044141,0.001948
1,LSTM,1003.371662,5.492276,0.017471,0.856802,0.046028,0.002119
2,Ridge,0.058098,0.005390,0.027568,0.809839,0.053042,0.002813


In [7]:
final_df.drop(columns=['MAE', 'MSE'], axis='columns', inplace=True)
final_df

,Model,Training Time (s),Prediction Time (s),R-squared,RMSE
0,XGBoost,1.064968,0.067575,0.868305,0.044141
1,LSTM,1003.371662,5.492276,0.856802,0.046028
2,Ridge,0.058098,0.005390,0.809839,0.053042


In [8]:
rmse_value = persistence_df.loc[0, 'RMSE']
rmse_value


0.0713702957793329

In [9]:
final_df['skill_score'] = 1 - (final_df['RMSE'] / rmse_value)
final_df

,Model,Training Time (s),Prediction Time (s),R-squared,RMSE,skill_score
0,XGBoost,1.064968,0.067575,0.868305,0.044141,0.381522
1,LSTM,1003.371662,5.492276,0.856802,0.046028,0.355075
2,Ridge,0.058098,0.005390,0.809839,0.053042,0.256808


In [10]:
ranking_df = pd.DataFrame()
ranking_df["Model"] = final_df["Model"]

for col in ["Training Time (s)", "Prediction Time (s)",  "RMSE"]:
    ranking_df[col] = final_df[col].rank(method="min", ascending=True).astype(int)

ranking_df["R-squared"] = final_df["R-squared"].rank(method="min", ascending=False).astype(int) 
ranking_df["skill_score"] = final_df["skill_score"].rank(method="min", ascending=False).astype(int) 

# Display the ranking table
ranking_df

,Model,Training Time (s),Prediction Time (s),RMSE,R-squared,skill_score
0,XGBoost,2,2,1,1,1
1,LSTM,3,3,2,2,2
2,Ridge,1,1,3,3,3


In [11]:
ranking_df["Row Average"] = ranking_df.iloc[:, 1:].mean(axis=1)

In [12]:
ranking_df.sort_values(by=['Row Average'])

,Model,Training Time (s),Prediction Time (s),RMSE,R-squared,skill_score,Row Average
0,XGBoost,2,2,1,1,1,1.4
2,Ridge,1,1,3,3,3,2.2
1,LSTM,3,3,2,2,2,2.4


**Min-Max Scaling**

In [13]:
def custom_minmax_normalization(df, higher_is_better, cutoff=-0.5):
    df_norm = pd.DataFrame(index=df.index)

    for col in df.columns:
        norm = df[col].copy()
        if col != 'Model':
            x = df[col].copy()

            x = x.clip(lower=cutoff)

            if higher_is_better[col]:
                norm = (x - x.min()) / (x.max() - x.min())
            else:
                norm = (x.max() - x) / (x.max() - x.min())

        df_norm[col] = norm

    return df_norm

def nested_harmonic_score(df, higher_is_better, cutoff=-0.5):
    df_norm = custom_minmax_normalization(df, higher_is_better, cutoff)

    # Performance Score: H(R2, RMSE, Skill Score)
    df_norm["Performance_Score"] = df_norm[["R-squared", "RMSE", "skill_score"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    # Speed Score: H(Train Time, Predict Time)
    df_norm["Speed_Score"] = df_norm[["Training Time (s)", "Prediction Time (s)"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    # Overall Score: H(Performance, Speed)
    df_norm["Overall_Score"] = df_norm[["Performance_Score", "Speed_Score"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    return df_norm[["Model", "Performance_Score", "Speed_Score", "Overall_Score"]]


higher_is_better = {
    "Training Time (s)": False,
    "Prediction Time (s)": False,
    "R-squared": True,
    "RMSE": False,
    "skill_score": True,
}

result = nested_harmonic_score(final_df, higher_is_better)
result.sort_values("Overall_Score", ascending=False, inplace=True)
result


,Model,Performance_Score,Speed_Score,Overall_Score
0,XGBoost,1.000000,0.993805,0.996893
1,LSTM,0.792976,0.000000,0.000000
2,Ridge,0.000000,1.000000,0.000000


**Rank-based normalization**

In [14]:
def custom_minmax_normalization(df):
    df_norm = pd.DataFrame(index=df.index)

    for col in df.columns:
        norm = df[col].copy()
        if col != 'Model':
            x = df[col].copy()

            norm = (x.max() - x) / (x.max() - x.min())

        df_norm[col] = norm

    return df_norm

def nested_harmonic_score(df):
    df_norm = custom_minmax_normalization(df)

    # Performance Score: H(R2, RMSE, Skill Score)
    df_norm["Performance_Score"] = df_norm[["R-squared", "RMSE", "skill_score"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    # Speed Score: H(Train Time, Predict Time)
    df_norm["Speed_Score"] = df_norm[["Training Time (s)", "Prediction Time (s)"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    # Overall Score: H(Performance, Speed)
    df_norm["Overall_Score"] = df_norm[["Performance_Score", "Speed_Score"]].apply(
        lambda row: hmean(row) if all(row > 0) else 0, axis=1
    )

    return df_norm[["Model", "Performance_Score", "Speed_Score", "Overall_Score"]]

ranking_df.drop(columns=['Row Average'], inplace=True)
result = nested_harmonic_score(ranking_df)
result.sort_values("Overall_Score", ascending=False, inplace=True)
result


,Model,Performance_Score,Speed_Score,Overall_Score
0,XGBoost,1.0,0.5,0.666667
1,LSTM,0.5,0.0,0.000000
2,Ridge,0.0,1.0,0.000000
